Hyperparameter Tuning

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import heapq
import itertools
import time
from enum import IntEnum
import os
%pip install optuna
import optuna
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

# ==========================================
# 1. CONFIGURATION
# ==========================================

class Actions(IntEnum):
    HOLD = 0
    BUY = 1
    SELL = 2

OBSERVATION_DIMS = 14
INITIAL_CASH = 100_000.0
MAX_STEPS_PER_EPISODE = 2000

# Fixed Env Params for Tuning
TRANSACTION_COST = 0.0 
INVENTORY_RISK_COEFF = 0.1
DOWNSIDE_PENALTY_MULT = 2.0

# ==========================================
# 2. CORE ENGINE
# ==========================================

class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# ==========================================
# 3. RL ENVIRONMENT
# ==========================================

class TradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super(TradingEnv, self).__init__()
        self.action_space = spaces.Discrete(len(Actions))
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(OBSERVATION_DIMS,), dtype=np.float32
        )
        self.engine = None
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [] 

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.engine = AdvancedOrderBook()
        start_price = 100.0
        for i in range(1, 6):
            self.engine.submit_order('buy', 10, start_price - i*0.5, 'limit')
            self.engine.submit_order('sell', 10, start_price + i*0.5, 'limit')
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [start_price] * 10 
        return self._get_observation(), {}

    def step(self, action):
        prev_val = self.portfolio_value
        
        # Action
        if action == Actions.BUY and self.cash > 0:
            rem = self.engine.submit_order('buy', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory += 1
                self.cash -= fill_price
        elif action == Actions.SELL:
            rem = self.engine.submit_order('sell', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory -= 1
                self.cash += abs(fill_price)

        # Market Move (High Volatility for Training)
        mid = self._get_mid_price()
        shock = np.random.normal(0, 2.0)
        self.engine.submit_order('buy', 10, round(mid + shock - 0.5, 2), 'limit')
        self.engine.submit_order('sell', 10, round(mid + shock + 0.5, 2), 'limit')

        # Update PnL
        mid = self._get_mid_price()
        self.portfolio_value = self.cash + (self.inventory * mid)
        if self.portfolio_value > self.max_portfolio_value:
            self.max_portfolio_value = self.portfolio_value
            
        # Reward
        delta_pnl = self.portfolio_value - prev_val
        inv_penalty = INVENTORY_RISK_COEFF * abs(self.inventory)
        dd_pct = (self.max_portfolio_value - self.portfolio_value) / self.max_portfolio_value
        dd_penalty = dd_pct * DOWNSIDE_PENALTY_MULT if dd_pct > 0.02 else 0
        
        reward = delta_pnl - inv_penalty - dd_penalty
        
        self.current_step += 1
        terminated = self.portfolio_value <= 0 
        truncated = self.current_step >= MAX_STEPS_PER_EPISODE
        
        return self._get_observation(), reward, terminated, truncated, {}

    def _get_mid_price(self):
        best_bid = -self.engine.bids[0][0] if self.engine.bids else 100.0
        best_ask = self.engine.asks[0][0] if self.engine.asks else 100.0
        mid = (best_bid + best_ask) / 2
        self.mid_price_history.append(mid)
        if len(self.mid_price_history) > 20: self.mid_price_history.pop(0)
        return mid

    def _get_observation(self):
        mid = self._get_mid_price()
        bids = [-x[0] for x in heapq.nsmallest(5, self.engine.bids)] if self.engine.bids else []
        asks = [x[0] for x in heapq.nsmallest(5, self.engine.asks)] if self.engine.asks else []
        while len(bids) < 5: bids.append(mid)
        while len(asks) < 5: asks.append(mid)
        
        norm_bids = [(p - mid)/mid for p in bids]
        norm_asks = [(p - mid)/mid for p in asks]
        norm_inv = self.inventory / 100.0
        norm_cash = self.cash / 1_000_000.0
        spread = (asks[0] - bids[0]) / mid
        vol = np.std(self.mid_price_history) / mid if len(self.mid_price_history) > 1 else 0
        
        return np.array(norm_bids + norm_asks + [norm_inv, norm_cash, spread, vol], dtype=np.float32)

# ==========================================
# 4. OPTUNA OPTIMIZATION LOOP
# ==========================================

def objective(trial):
    """
    Optuna optimization function.
    1. Sample hyperparameters.
    2. Train model.
    3. Return evaluation score.
    """
    # 1. Sample Hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    gamma = trial.suggest_float("gamma", 0.9, 0.9999)
    ent_coef = trial.suggest_float("ent_coef", 0.001, 0.1, log=True)
    
    # 2. Setup Environment
    # We use a vectorized env for speed, but single env is fine too
    env = make_vec_env(lambda: TradingEnv(), n_envs=1)
    
    # 3. Setup Model
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        gamma=gamma,
        ent_coef=ent_coef,
        verbose=0
    )
    
    # 4. Train (Short run for tuning: 10k steps)
    # Real tuning would use 100k+, but 10k is enough to see learning speed differences
    try:
        model.learn(total_timesteps=10000)
    except AssertionError:
        # Sometimes PPO crashes with bad params
        return -float('inf')
    
    # 5. Evaluate
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=5)
    
    return mean_reward

def run_tuning():
    print("--- Week 3 Day 9: Hyperparameter Tuning with Optuna ---")
    print("Searching for the best Learning Rate, Gamma, and Entropy...")
    
    # Create study
    study = optuna.create_study(direction="maximize")
    
    # Run 10 trials (You can increase this if you have time/GPU)
    study.optimize(objective, n_trials=10)
    
    print("\n--- Tuning Complete ---")
    print("Best params found:")
    print(study.best_params)
    print(f"Best Reward: {study.best_value}")
    
    # Save best params to file for Day 10
    with open("best_params.txt", "w") as f:
        f.write(str(study.best_params))
        
    # Visualize optimization history (Optional)
    try:
        fig = optuna.visualization.plot_optimization_history(study)
        fig.write_image("optimization_history.png")
        print("Saved optimization_history.png")
    except:
        pass

if __name__ == "__main__":
    run_tuning()

  Using cached optuna-4.7.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
Using cached optuna-4.7.0-py3-none-any.whl (413 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
Note: you may need to restart the kernel to use updated packages.


[I 2026-02-04 02:11:07,044] A new study created in memory with name: no-name-dec184e5-6fee-4757-bb2e-85426825eea3


--- Week 3 Day 9: Hyperparameter Tuning with Optuna ---
Searching for the best Learning Rate, Gamma, and Entropy...


[I 2026-02-04 02:11:23,133] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.0001731120248406722, 'gamma': 0.9731499610680441, 'ent_coef': 0.0812779239787475}. Best is trial 0 with value: 0.0.
[I 2026-02-04 02:11:36,919] Trial 1 finished with value: 0.0 and parameters: {'learning_rate': 0.00022357998564857609, 'gamma': 0.9280026102083554, 'ent_coef': 0.006369090856485631}. Best is trial 0 with value: 0.0.
[I 2026-02-04 02:11:50,566] Trial 2 finished with value: -1193.946 and parameters: {'learning_rate': 4.936528120682783e-05, 'gamma': 0.9903443442682878, 'ent_coef': 0.015536084176749796}. Best is trial 0 with value: 0.0.
[I 2026-02-04 02:12:04,527] Trial 3 finished with value: -4372.782 and parameters: {'learning_rate': 0.00022709628161177346, 'gamma': 0.9198998998129951, 'ent_coef': 0.0012264924529272589}. Best is trial 0 with value: 0.0.
[I 2026-02-04 02:12:18,178] Trial 4 finished with value: 0.0 and parameters: {'learning_rate': 1.5543241493736273e-05, 'gamma':


--- Tuning Complete ---
Best params found:
{'learning_rate': 0.0001731120248406722, 'gamma': 0.9731499610680441, 'ent_coef': 0.0812779239787475}
Best Reward: 0.0
